# Chapter 7: Design a Unique ID Generator
*System Design Interview*

## TL;DR

For unique, time-sortable, 64-bit IDs at scale, **Twitter Snowflake** is the go-to: 1 sign bit + 41 timestamp bits + 5 datacenter bits + 5 machine bits + 12 sequence bits. It supports ~4096 IDs/ms/machine, is decentralized, and IDs are naturally sortable by time. Alternatives (UUID, ticket server, multi-master) each have significant trade-offs.

## Requirements

| Requirement | Detail |
|---|---|
| Uniqueness | IDs must be globally unique |
| Numeric only | IDs are numerical values |
| 64-bit | Must fit in 64 bits |
| Time-ordered | IDs created later have larger values |
| Throughput | > 10,000 IDs per second |

## Estimation: Snowflake Capacity

In [ ]:
# Twitter Snowflake bit allocation
sign_bits = 1
timestamp_bits = 41
datacenter_bits = 5
machine_bits = 5
sequence_bits = 12

total = sign_bits + timestamp_bits + datacenter_bits + machine_bits + sequence_bits
print(f'Total bits: {total}')

max_datacenters = 2 ** datacenter_bits
max_machines = 2 ** machine_bits
max_sequence = 2 ** sequence_bits

print(f'Max datacenters: {max_datacenters}')
print(f'Max machines per DC: {max_machines}')
print(f'Max IDs per ms per machine: {max_sequence}')
print(f'Max IDs per second per machine: {max_sequence * 1000:,}')
print(f'Max total machines: {max_datacenters * max_machines}')
print(f'Max IDs/sec (all machines): {max_sequence * 1000 * max_datacenters * max_machines:,}')

## Estimation: Snowflake Timestamp Lifespan

In [ ]:
# How long until 41-bit timestamp overflows?
max_ms = 2**41 - 1
years = max_ms / 1000 / 3600 / 24 / 365.25
print(f'Max milliseconds in 41 bits: {max_ms:,}')
print(f'That is approximately {years:.1f} years')
print(f'\nWith custom epoch starting 2020-01-01, overflow around {2020 + years:.0f}')

## Estimation: Comparing ID Generation Approaches

In [ ]:
# Quick comparison of ID generation approaches
approaches = [
    ('Multi-master (k=5)', 64, True, False, 'auto_increment by k'),
    ('UUID',               128, False, True, 'random, no coordination'),
    ('Ticket server',      64, True, False, 'centralized counter'),
    ('Snowflake',          64, True, True, 'timestamp + machine + seq'),
]

print(f'{"Approach":<25} {"Bits":>4}  {"Sorted?":<8} {"Decentralized?":<15} Notes')
print('-' * 90)
for name, bits, sortable, decentralized, notes in approaches:
    s = 'Yes' if sortable else 'No'
    d = 'Yes' if decentralized else 'No'
    print(f'{name:<25} {bits:>4}  {s:<8} {d:<15} {notes}')

## High-Level Design

```mermaid
graph LR
    subgraph Snowflake ID 64 bits
        A[Sign 1b] --> B[Timestamp 41b]
        B --> C[Datacenter 5b]
        C --> D[Machine 5b]
        D --> E[Sequence 12b]
    end
```

```mermaid
graph TB
    App1[App Server] --> IDGen1[ID Generator DC1-M1]
    App2[App Server] --> IDGen2[ID Generator DC1-M2]
    App3[App Server] --> IDGen3[ID Generator DC2-M1]
    IDGen1 --> DB[(Database)]
    IDGen2 --> DB
    IDGen3 --> DB
```

## Deep Dive

### Multi-Master Replication
Increment by k (number of DB servers) instead of 1. Simple but IDs are not time-ordered across servers, and adding/removing servers changes k.

### UUID
128-bit, generated independently, near-zero collision. But too long (need 64-bit), not time-sortable, and can be non-numeric.

### Ticket Server
Centralized auto-increment (Flickr). Simple and numeric, but a **single point of failure**. Multiple ticket servers introduce synchronization complexity.

### Twitter Snowflake (chosen approach)
- **Timestamp** (41 bits): milliseconds since custom epoch, makes IDs time-sortable
- **Datacenter ID** (5 bits): 32 datacenters
- **Machine ID** (5 bits): 32 machines per DC
- **Sequence** (12 bits): 4096 IDs/ms/machine, resets each ms

### Clock Synchronization
Snowflake depends on synchronized clocks. NTP (Network Time Protocol) is the standard solution. Clock drift can cause duplicate or out-of-order IDs.

### Section Length Tuning
- Low concurrency, long lifespan: fewer sequence bits, more timestamp bits
- High concurrency, short lifespan: more sequence bits, fewer timestamp bits

## Trade-offs

| Approach | Pros | Cons |
|---|---|---|
| Multi-master | Simple, uses existing DB | Not time-sorted, hard to scale |
| UUID | No coordination needed | 128 bits, not sortable, non-numeric |
| Ticket server | Simple, numeric | Single point of failure |
| Snowflake | 64-bit, time-sorted, decentralized | Depends on clock sync |

## Key Takeaways

1. **Snowflake** is the best fit for 64-bit, time-sortable, distributed IDs
2. 41 timestamp bits give ~69 years of lifespan from a custom epoch
3. 12 sequence bits allow 4096 IDs/ms/machine (4M+ IDs/sec)
4. Clock synchronization (NTP) is a critical operational dependency
5. Bit allocation is tunable based on concurrency vs lifespan needs

## Related Concepts

- [[multi-master-replication]] -- auto-increment with step k
- [[uuid]] -- decentralized 128-bit identifiers
- [[ticket-server]] -- centralized ID issuing
- [[twitter-snowflake]] -- the 64-bit time-sortable scheme
- [[clock-synchronization]] -- NTP and the clock drift problem